# Health Vault: Standalone OCR Engine Comparison & Debugger

This standalone notebook provides an isolated diagnostic and benchmarking harness for evaluating the two Health Vault OCR engines on the same document:
1. **PaddleOCR** (CPU-based primary OCR)
2. **VLM / Qwen3-VL** (Multimodal Vision Language Model on Ollama)

### Isolation & Execution Notice:
- **No Automatic Fallbacks**: The production `QualityGate` and automatic `PaddleOCR -> VLM` fallback are **not** invoked in this notebook.
- **Explicit Engine Selection**: You choose which engine to run via `ENGINE = "paddleocr"` or `ENGINE = "vlm"` in the configuration cell below.
- **Side-by-Side Diagnostic**: Inspect raw model outputs, per-page latency, character density, line-by-line bounding boxes, and transcription quality independently.


In [ ]:
# ==============================================================================
# 1. USER CONFIGURATION
# ==============================================================================
DIRECTORY = r"D:\TECHROVER\PROJECTS\RTH\healh-vault-project\Medical Reports"
FILENAME = "60300505154.pdf"

ENGINE = "paddleocr"   # Options: "paddleocr" or "vlm"

PDF_DPI = 150
APPLY_PREPROCESSING = False  # Set to True to apply OpenCV deskew and contrast enhancement


In [ ]:
# ==============================================================================
# 2. ENVIRONMENT SETUP & CORE IMPORTS
# ==============================================================================
import os
import sys
import io
import time
import json
import logging
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

# Ensure ai-service root is in sys.path
NOTEBOOK_DIR = Path.cwd()
AI_SERVICE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(AI_SERVICE_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_DIR))

# Safely pre-load PyTorch DLLs on Windows before Paddle/OpenMP initializes
try:
    import torch  # noqa: F401
except Exception:
    pass

from app.settings import get_settings
from app.modules.ocr.paddle_engine import PaddleOcrEngine
from app.modules.vision.vision_service import VisionModelService, _OCR_PROMPT, _parse_json, _normalize_pages
from app.services.pipeline.preprocessing import preprocess_document_image
from app.modules.ocr.cleanup import clean_ocr_text

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("ocr_comparison")

print("Environment initialized successfully.")
print(f"Selected OCR Engine: {ENGINE.upper()}")


In [ ]:
# ==============================================================================
# 3. DOCUMENT INGESTION & PAGE RENDERING
# ==============================================================================
import fitz

target_path = Path(DIRECTORY) / FILENAME

if not target_path.exists():
    fallback_path = AI_SERVICE_DIR / "tests" / "fixtures" / "golden_docs" / "Manjuben Ranoliya.pdf"
    print(f"Target file not found at: {target_path}")
    print(f"Falling back to golden fixture: {fallback_path.name}")
    target_path = fallback_path

print(f"Loading document: {target_path}")
file_bytes = target_path.read_bytes()
file_ext = target_path.suffix.lower()

pages = []  # List of tuples: (page_num, image_bytes, width, height)

if file_ext == ".pdf":
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    print(f"Opened PDF with {doc.page_count} page(s). Rendering at {PDF_DPI} DPI...")
    scale = PDF_DPI / 72.0
    matrix = fitz.Matrix(scale, scale)
    for idx in range(doc.page_count):
        page = doc.load_page(idx)
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        img_bytes = pix.tobytes("jpeg", jpg_quality=88)
        pages.append((idx + 1, img_bytes, pix.width, pix.height))
    doc.close()
elif file_ext in (".tiff", ".tif"):
    from PIL import ImageSequence
    with Image.open(io.BytesIO(file_bytes)) as img:
        for idx, frame in enumerate(ImageSequence.Iterator(img)):
            frame_rgb = frame.convert("RGB")
            buf = io.BytesIO()
            frame_rgb.save(buf, format="JPEG", quality=88)
            pages.append((idx + 1, buf.getvalue(), frame.width, frame.height))
else:
    # Single image (PNG, JPG, WEBP)
    with Image.open(io.BytesIO(file_bytes)) as img:
        w, h = img.size
    pages.append((1, file_bytes, w, h))

print(f"Rendered {len(pages)} page(s) ready for OCR extraction.")
for p_num, p_bytes, w, h in pages:
    print(f"  * Page {p_num}: {w}x{h} px ({len(p_bytes):,} bytes)")

# Display preview thumbnail of Page 1
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 8))
    p1_img = Image.open(io.BytesIO(pages[0][1]))
    plt.imshow(p1_img)
    plt.title(f"{target_path.name} - Page 1 Preview ({p1_img.width}x{p1_img.height})")
    plt.axis("off")
    plt.show()
except Exception as e:
    print(f"Image preview skipped: {e}")


In [ ]:
# ==============================================================================
# 4. EXECUTION PATH 1: PADDLEOCR (ISOLATED - NO FALLBACK)
# ==============================================================================
paddle_results = []

if ENGINE.strip().lower() == "paddleocr":
    print("=" * 70)
    print("RUNNING PADDLEOCR IN ISOLATION")
    print("=" * 70)
    
    t_init = time.monotonic()
    engine = PaddleOcrEngine.get_instance(lang="en")
    print(f"PaddleOcrEngine ready on device: {engine.device} (init: {(time.monotonic() - t_init)*1000:.1f}ms)")
    
    for p_num, p_bytes, w, h in pages:
        if APPLY_PREPROCESSING:
            print(f"Applying OpenCV preprocessing to Page {p_num}...")
            p_bytes = preprocess_document_image(p_bytes, deskew=True, remove_shadows=True, binarize=False)
            
        print(f"Extracting Page {p_num}/{len(pages)} via PaddleOCR...")
        t0 = time.monotonic()
        res = engine.extract_text_from_bytes(p_bytes)
        elapsed_s = time.monotonic() - t0
        
        paddle_results.append({
            "page": p_num,
            "engine": "paddleocr",
            "device": engine.device,
            "elapsed_sec": round(elapsed_s, 3),
            "line_count": res.get("line_count", 0),
            "char_count": res.get("char_count", 0),
            "mean_confidence": res.get("mean_confidence", 0.0),
            "min_confidence": res.get("min_confidence", 0.0),
            "full_text": res.get("full_text", ""),
            "lines": res.get("lines", []),
            "status": "SUCCESS" if res.get("char_count", 0) > 0 else "EMPTY",
        })
        print(f"  Page {p_num} complete: {res.get('line_count')} lines, {res.get('char_count')} chars, conf={res.get('mean_confidence'):.4f} in {elapsed_s:.2f}s")
else:
    print(f"Skipping PaddleOCR execution (ENGINE is set to '{ENGINE}')")


In [ ]:
# ==============================================================================
# 5. EXECUTION PATH 2: VLM / QWEN3-VL (ISOLATED - NO FALLBACK)
# ==============================================================================
vlm_results = []

if ENGINE.strip().lower() == "vlm":
    print("=" * 70)
    print("RUNNING VLM (QWEN3-VL) IN ISOLATION")
    print("=" * 70)
    
    settings = get_settings()
    vision_service = VisionModelService(
        api_key=settings.ai_api_key or "",
        base_url=settings.ai_base_url,
        model=settings.ai_model,
        timeout_seconds=settings.ai_timeout_seconds,
        max_retries=settings.ai_max_retries,
        max_output_tokens=settings.ai_max_output_tokens,
        min_text_chars=settings.ai_min_text_chars,
        cache_size=0,  # Zero cache for clean benchmarking
        max_inline_bytes=settings.ai_max_inline_bytes,
        page_concurrency=1,
    )
    
    print(f"Target Vision Model: {vision_service.model} via {vision_service.base_url}")
    client = vision_service._ensure_client()
    
    async def run_vlm_extraction():
        for p_num, p_bytes, w, h in pages:
            if APPLY_PREPROCESSING:
                print(f"Applying OpenCV preprocessing to Page {p_num}...")
                p_bytes = preprocess_document_image(p_bytes, deskew=True, remove_shadows=True, binarize=False)
            
            print(f"Sending Page {p_num}/{len(pages)} to {vision_service.model}...")
            t0 = time.monotonic()
            raw_response, finish_reason = await client.generate_json_from_bytes(
                data=p_bytes,
                mime_type="image/jpeg",
                prompt=_OCR_PROMPT,
            )
            elapsed_s = time.monotonic() - t0
            
            # Parse structured response
            parsed = _parse_json(raw_response)
            extracted_pages = None
            if isinstance(parsed, dict):
                if isinstance(parsed.get("pages"), list):
                    extracted_pages = _normalize_pages(parsed["pages"])
                elif "text" in parsed and isinstance(parsed["text"], str):
                    extracted_pages = _normalize_pages([{"page": p_num, "text": parsed["text"], "confidence": parsed.get("confidence", 0.95)}])
            
            # Plain-text fallback if unformatted
            if not extracted_pages:
                import re
                plain_txt = re.sub(r"<think>.*?</think>", "", raw_response or "", flags=re.DOTALL).strip()
                extracted_pages = _normalize_pages([{"page": p_num, "text": plain_txt, "confidence": 0.85}])
                
            page_obj = extracted_pages[0] if extracted_pages else {"text": "", "confidence": 0.0}
            text = clean_ocr_text(page_obj.get("text", ""))
            lines = [l.strip() for l in text.splitlines() if l.strip()]
            
            vlm_results.append({
                "page": p_num,
                "engine": f"vlm:{vision_service.model}",
                "elapsed_sec": round(elapsed_s, 3),
                "line_count": len(lines),
                "char_count": len(text),
                "mean_confidence": page_obj.get("confidence") or 0.95,
                "finish_reason": finish_reason,
                "raw_response": raw_response,
                "full_text": text,
                "lines": [{"text": l, "confidence": page_obj.get("confidence")} for l in lines],
                "status": "SUCCESS" if len(text) > 0 else "EMPTY",
            })
            print(f"  Page {p_num} complete: {len(lines)} lines, {len(text)} chars in {elapsed_s:.2f}s (finish: {finish_reason})")
    
    import asyncio
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import concurrent.futures
            with concurrent.futures.ThreadPoolExecutor() as pool:
                pool.submit(asyncio.run, run_vlm_extraction()).result()
        else:
            loop.run_until_complete(run_vlm_extraction())
    except Exception:
        asyncio.run(run_vlm_extraction())
else:
    print(f"Skipping VLM execution (ENGINE is set to '{ENGINE}')")


In [ ]:
# ==============================================================================
# 6. DIAGNOSTIC RESULTS & SUMMARY TABLE
# ==============================================================================
active_results = paddle_results if ENGINE.strip().lower() == "paddleocr" else vlm_results

if active_results:
    summary_df = pd.DataFrame([
        {
            "Page": r["page"],
            "Engine": r["engine"],
            "Duration (s)": r["elapsed_sec"],
            "Characters": r["char_count"],
            "Lines": r["line_count"],
            "Mean Confidence": f"{r['mean_confidence']:.4f}",
            "Status": r["status"],
        }
        for r in active_results
    ])
    
    total_time = sum(r["elapsed_sec"] for r in active_results)
    total_chars = sum(r["char_count"] for r in active_results)
    total_lines = sum(r["line_count"] for r in active_results)
    
    print("\n" + "=" * 70)
    print(f"EXTRACTION SUMMARY: {ENGINE.upper()}")
    print("=" * 70)
    print(f"Total Pages: {len(active_results)}")
    print(f"Total Duration: {total_time:.2f}s (avg: {total_time/len(active_results):.2f}s/page)")
    print(f"Total Characters: {total_chars:,}")
    print(f"Total Lines: {total_lines:,}")
    print("=" * 70)
    
    display(summary_df)
else:
    print(f"No results recorded for ENGINE='{ENGINE}'. Please check cell 1 configuration.")


In [ ]:
# ==============================================================================
# 7. GRANULAR DEBUGGER & TEXT INSPECTION
# ==============================================================================
if active_results:
    for r in active_results:
        print(f"\n{'='*70}\nPAGE {r['page']} TEXT EXTRACTION ({r['engine']})\n{'='*70}")
        print(r["full_text"])
        
        if r.get("lines") and len(r["lines"]) > 0:
            print(f"\n--- Line-by-Line Inspection (First 15 lines of Page {r['page']}) ---")
            lines_df = pd.DataFrame(r["lines"][:15])
            display(lines_df)
            
        if "raw_response" in r and r["raw_response"]:
            print(f"\n--- Raw Model Output (First 500 chars) ---")
            print(r["raw_response"][:500] + ("..." if len(r["raw_response"]) > 500 else ""))
